# Activity — AI Agent Backend Architecture

## Lesson — Building the Backend of an AI Agent Application

In this activity, you will build a simple backend for an AI agent application.

You will learn how the different components of an AI agent system work together:

```text
User
  ↓
Frontend
  ↓
API
  ↓
Agent
  ↓
Tools / RAG / Memory
  ↓
External Systems




## Learning Objectives

By the end of this activity, you will be able to:

- Understand the basic backend architecture of an AI agent
- Create a simple API using FastAPI
- Connect an API to an AI model
- Use environment variables for configuration
- Select and configure an AI model
- Measure API latency
- Handle API errors
- Understand rate limits
- Understand the basic factors that affect AI application cost
- Connect tools and memory to an agent architecture

## Architecture

The application we will build follows this architecture:

```text
User
  ↓
API Client
  ↓
FastAPI
  ↓
Agent
  ↓
Tools / Memory
  ↓
External System


## Step 1 — Install the Required Libraries

We will use:

- `google-genai` for Gemini
- `fastapi` for our backend API
- `uvicorn` for running FastAPI
- `python-dotenv` for environment variables

In [1]:
!pip -q install google-genai fastapi uvicorn python-dotenv

### Step 2 — Import Libraries

In [2]:
import os
import time

from google import genai
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

## Step 3 — Configure the API Key

API keys are sensitive information.

We should not write API keys directly inside our source code.

Instead, we can load them through environment variables.

For this Colab activity, we will enter the key securely using `getpass`.

In [ ]:
import getpass

api_key = getpass.getpass("Enter your Gemini API key: ")

os.environ["GEMINI_API_KEY"] = api_key

print("API key loaded successfully.")

## Step 4 — Model Selection

The model is an important part of an AI application's architecture.

When selecting a model, we should consider:

- Capability
- Cost
- Latency
- Context length
- Tool-calling support
- Rate limits
- Availability

For this activity, we will use a Gemini Flash model.

In [ ]:
MODEL = "gemini-3.8-flash"

print("Selected model:", MODEL)

## Step 5 — Create the Gemini Client

In [ ]:
client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

print("Gemini client created successfully.")

## Step 6 — Test the Model

Before connecting Gemini to our backend, let's make sure the model is working.

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="Explain what an AI agent is in one sentence."
)

print(response.text)

## Step 7 — Create the Agent

The agent is the reasoning component of our backend.

For this activity, we will keep the agent simple.

It receives a user message and sends it to the selected AI model.

Later, an agent can be extended with:

- Tools
- RAG
- Memory
- APIs
- Databases
- External systems

In [ ]:
def run_agent(user_message):

    response = client.models.generate_content(
        model=MODEL,
        contents=user_message
    )

    return response.text

# Step 8 — Test the Agent

In [ ]:
answer = run_agent(
    "What is the purpose of an API in an AI application?"
)

print("Agent response:")
print(answer)

## Step 9 — Add Memory

An AI agent can use memory to keep track of previous interactions.

For this simple activity, we will use a Python list as short-term conversation memory.

In a production application, memory could be stored in:

- A database
- Redis
- A vector database
- A conversation store

In [ ]:
conversation_history = []

print("Memory initialized.")

## Step 10 — Add Conversation Memory to the Agent

The agent will now receive previous messages along with the new user message.

This allows the model to use information from earlier interactions.

In [ ]:
def run_agent_with_memory(user_message):

    conversation_history.append(
        {
            "role": "user",
            "content": user_message
        }
    )

    conversation_text = ""

    for message in conversation_history:
        conversation_text += (
            f"{message['role']}: {message['content']}\n"
        )

    response = client.models.generate_content(
        model=MODEL,
        contents=conversation_text
    )

    answer = response.text

    conversation_history.append(
        {
            "role": "assistant",
            "content": answer
        }
    )

    return answer

## Step 11 — Test Memory

Let's send two messages.

The second message should have access to the previous conversation.

In [ ]:
answer = run_agent_with_memory(
    "My favorite programming language is Python."
)

print(answer)

In [ ]:
answer = run_agent_with_memory(
    "What programming language did I mention?"
)

print(answer)

## Step 12 — Add a Tool

Agents can use tools to perform tasks outside the model itself.

Examples include:

- Calculator
- Web search
- Database lookup
- File search
- External APIs

For this activity, we will create a simple system-information tool.

In [ ]:
def get_system_info():

    return {
        "application": "AI Agent Backend",
        "environment": "Google Colab",
        "backend": "FastAPI",
        "model": MODEL
    }

In [ ]:
system_info = get_system_info()

print(system_info)

## Step 13 — External Systems

In a real application, a tool may communicate with an external system.

For example:

```text
Agent
  ↓
Tool
  ↓
External API
  ↓
External System


## Step 14 — Build the FastAPI Backend

Now we will create the API layer.

The API will receive a user message and send it to the agent.

Our architecture is now:

```text
User
  ↓
API Client
  ↓
FastAPI
  ↓
Agent
  ↓
Gemini

In [ ]:
app = FastAPI(
    title="AI Agent Backend",
    description="A simple backend for an AI agent",
    version="1.0"
)

print("FastAPI application created.")

## Step 15 — Define the API Request

We need to define what data the API expects from the frontend.

Our `/chat` endpoint will receive a user message.

In [ ]:
class ChatRequest(BaseModel):
    message: str

## Step 16 — Create the Chat API

The `/chat` endpoint will:

1. Receive the user's message
2. Send it to the agent
3. Receive the agent response
4. Return the response to the client

In [ ]:
@app.post("/chat")
def chat(request: ChatRequest):

    try:

        answer = run_agent(
            request.message
        )

        return {
            "response": answer
        }

    except Exception as e:

        raise HTTPException(
            status_code=500,
            detail=f"Agent error: {str(e)}"
        )

## Step 17 — Add a Health Check

Backend applications commonly have a health-check endpoint.

It can be used to verify that the backend is running.

In [ ]:
@app.get("/")
def health_check():

    return {
        "status": "running",
        "service": "AI Agent Backend"
    }

## Step 18 — Test the API

Instead of deploying the API, we can test it directly inside Colab.

We will use FastAPI's `TestClient`.

This allows us to simulate frontend requests.

In [ ]:
from fastapi.testclient import TestClient

api_client = TestClient(app)

print("API test client created.")

In [ ]:
response = api_client.get("/")

print("Status code:", response.status_code)
print("Response:", response.json())

## Step 19 — Test the Chat API

Now we will simulate a frontend sending a message to our backend.

The request flow is:

```text
Frontend
   ↓
POST /chat
   ↓
FastAPI
   ↓
Agent
   ↓
Gemini
   ↓
Response

In [ ]:
response = api_client.post(
    "/chat",
    json={
        "message": "Explain what an AI agent is."
    }
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

## Step 20 — Measure Latency

Latency is the amount of time required to process a request.

For an AI application, latency can come from:

- API processing
- Model response time
- Tool calls
- Database queries
- External APIs
- Network requests

Let's measure the time required for our API request.

In [ ]:
start_time = time.time()

response = api_client.post(
    "/chat",
    json={
        "message": "What is machine learning?"
    }
)

end_time = time.time()

latency = end_time - start_time

print("Status code:", response.status_code)
print("Latency:", round(latency, 2), "seconds")
print("Response:", response.json())

## Step 21 — Error Handling

AI applications can fail for many reasons.

Common problems include:

- Invalid API keys
- Model unavailable
- Rate limits
- Network errors
- Tool failures
- Database failures
- Invalid user input

Our API uses `try` and `except` to prevent an internal error from crashing the entire application.

In [ ]:
@app.post("/chat-safe")
def safe_chat(request: ChatRequest):

    if not request.message.strip():

        raise HTTPException(
            status_code=400,
            detail="Message cannot be empty."
        )

    try:

        answer = run_agent(
            request.message
        )

        return {
            "response": answer
        }

    except Exception as e:

        return {
            "error": "The agent could not process the request.",
            "details": str(e)
        }

## Step 22 — Test Input Validation

Let's test what happens when the user sends an empty message.

In [ ]:
response = api_client.post(
    "/chat-safe",
    json={
        "message": ""
    }
)

print("Status code:", response.status_code)
print("Response:", response.json())

## Step 23 — Rate Limits

AI APIs may limit how many requests can be made within a specific period.

A common response is:

```text
429 Too Many Requests


## Step 24 — Simple Retry Strategy

We can create a helper that retries temporary API failures.

The goal is not to retry every error.

Retries are useful for temporary problems such as service unavailability or rate limits.


In [ ]:
def generate_with_retry(
    prompt,
    max_retries=3,
    delay=2
):

    for attempt in range(max_retries):

        try:

            response = client.models.generate_content(
                model=MODEL,
                contents=prompt
            )

            return response.text

        except Exception as e:

            print(
                f"Request failed. "
                f"Retry {attempt + 1}/{max_retries}"
            )

            if attempt == max_retries - 1:
                raise e

            time.sleep(delay * (2 ** attempt))

## Step 25 — Cost

AI application cost can depend on several factors:

```text
Number of users
      ×
Requests per user
      ×
Model usage
      ×
Tokens used
      ×
Number of model/tool calls


## Step 26 — Model Selection

Model selection is an engineering decision.

A more capable model may provide better results but may also have higher cost or latency.

When selecting a model, consider:

| Factor | Question |
|---|---|
| Capability | Can the model perform the required task? |
| Cost | Is the model affordable at the expected scale? |
| Latency | Is the response fast enough? |
| Context | Can it handle the required information? |
| Tools | Does it support the required tool interactions? |
| Availability | Is the model reliably available? |

There is no single model that is best for every application.

## Step 27 — Complete Architecture

We have now implemented the main parts of our backend.

The complete architecture is:

```text
                    User
                      ↓
              Frontend / Client
                      ↓
                    FastAPI
                      ↓
                    Agent
                  ↙   ↓   ↘
              Tools  Memory  Model
                ↓      ↓      ↓
          External   History  Gemini
           Systems


## Step 28 — Architecture Review

Let's connect each topic from this activity to the application we built.

### Backend Architecture

FastAPI provides the backend API layer.

### APIs

The `/chat` endpoint allows clients to communicate with the agent.

### Model Selection

The `MODEL` variable determines which AI model is used.

### Environment Variables

The Gemini API key is stored as an environment variable.

### Rate Limits

The application needs retry and backoff strategies for temporary API limits and failures.

### Cost

Model calls, tokens, users, and agent steps affect application cost.

### Latency

Model calls and external operations contribute to response time.

### Error Handling

The API catches errors and returns appropriate responses.

# Key Takeaways

In this activity, you built a basic backend architecture for an AI agent application.

You learned how:

- A frontend communicates with an API
- An API communicates with an agent
- An agent communicates with an AI model
- Tools can provide additional capabilities
- Memory can store conversation history
- External systems can provide additional data
- Environment variables protect configuration and secrets
- Model selection affects cost and latency
- Rate limits can affect application reliability
- Error handling helps prevent application failures

The overall architecture is:

```text
User
  ↓
Frontend
  ↓
API
  ↓
Agent
  ↓
Tools / RAG / Memory
  ↓
External Systems


---

## One important change for your course

I would **not add a student challenge inside this notebook**, just like we decided for Lesson 4.

This Colab should be the **complete guided implementation**.

Later, you can create a separate student `.py` activity where they have to build things themselves, for example:

```text
Activity-05-AI-Agent-System-Architecture/
│
├── README.md
└── activity.py